# Bài 4: Huấn luyện mô hình cho bộ dữ liệu `kc_house_data` theo pipeline

**Sinh viên:** Nguyễn Đức Duy · **MSV:** 24174800068 · **Lớp:** DHKM18A2HN

**Pipeline:** Raw Data → Data Cleaning → EDA → Feature Engineering → Feature Selection → Train/Test Split → Model Training → Prediction → Evaluation

Bài toán: dự đoán giá nhà (`price`). 8 mô hình × các giai đoạn: baseline, log-target, Feature Engineering, Feature Selection, tinh chỉnh siêu tham số. Đánh giá bằng MAE, RMSE, R², MAPE (luôn tính trên thang giá USD gốc).

> Lưu ý: notebook chạy đầy đủ mất khoảng 10 đến 15 phút (Random Forest, Gradient Boosting, cross-validation và tìm tham số). Cần đặt `kc_house_data.csv` cùng thư mục.

## 1. Thư viện, đọc dữ liệu và làm sạch (Raw Data → Data Cleaning)

In [6]:
import numpy as np, pandas as pd, json, time, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, RFE, SelectFromModel
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
SAVE = False  # đặt True nếu muốn lưu các biểu đồ ra file png

In [2]:
PATH = "kc_house_data.csv"   # đặt file csv cùng thư mục với notebook
SEED = 42; t0 = time.time()
def log(*a): print(f"[{time.time()-t0:5.0f}s]", *a, flush=True)

# ---------- 1. Du lieu & lam sach ----------
df = pd.read_csv(PATH)
n0 = len(df)
df = df[df["bedrooms"] != 33].reset_index(drop=True)      # loi nhap lieu: 33 phong ngu / 1620 sqft
df["date"] = pd.to_datetime(df["date"].str[:8], format="%Y%m%d")
y = df["price"].copy()
log("rows", n0, "->", len(df))

[    0s] rows 21613 -> 21612


In [3]:
print("Kích thước:", df.shape)
print("Giá trị thiếu:", df.isnull().sum().sum(), "| dòng trùng hoàn toàn:", df.duplicated().sum(), "| id lặp (bán lại):", df["id"].duplicated().sum())
display(df.drop(columns=["date"]).describe().T.round(2))

Kích thước: (21612, 21)
Giá trị thiếu: 0 | dòng trùng hoàn toàn: 0 | id lặp (bán lại): 177


,count,mean,std,min,25%,50%,75%,max
id,21612.0,4.580402e+09,2.876594e+09,1000102.00,2.123049e+09,3.904930e+09,7.308900e+09,9.900000e+09
price,21612.0,5.400835e+05,3.671351e+05,75000.00,3.218375e+05,4.500000e+05,6.450000e+05,7.700000e+06
bedrooms,21612.0,3.370000e+00,9.100000e-01,0.00,3.000000e+00,3.000000e+00,4.000000e+00,1.100000e+01
bathrooms,21612.0,2.110000e+00,7.700000e-01,0.00,1.750000e+00,2.250000e+00,2.500000e+00,8.000000e+00
sqft_living,21612.0,2.079920e+03,9.184600e+02,290.00,1.426500e+03,1.910000e+03,2.550000e+03,1.354000e+04
sqft_lot,21612.0,1.510739e+04,4.142142e+04,520.00,5.040000e+03,7.619000e+03,1.068825e+04,1.651359e+06
floors,21612.0,1.490000e+00,5.400000e-01,1.00,1.000000e+00,1.500000e+00,2.000000e+00,3.500000e+00
waterfront,21612.0,1.000000e-02,9.000000e-02,0.00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
view,21612.0,2.300000e-01,7.700000e-01,0.00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00
condition,21612.0,3.410000e+00,6.500000e-01,1.00,3.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00


## 2. Khám phá dữ liệu (EDA)

- Không có giá trị thiếu. Bản ghi `bedrooms = 33` (chỉ 1.620 sqft) là lỗi nhập liệu nên bị loại.
- `price` lệch phải mạnh (skew ≈ 4,0) → thử biến đổi `log(price)` (skew ≈ 0,43).
- Các outlier giá cao là nhà sang thật, được giữ lại.

In [4]:
BASE = ["bedrooms","bathrooms","sqft_living","sqft_lot","floors","waterfront","view","condition","grade",
        "sqft_above","sqft_basement","yr_built","yr_renovated","zipcode","lat","long","sqft_living15","sqft_lot15"]
corr_price = df[BASE].assign(price=y).corr()["price"].drop("price").sort_values(key=abs, ascending=False)
display(corr_price.to_frame("corr_with_price").round(3))
q1, q3 = y.quantile([.25, .75]); print("Số giá nhà outlier theo IQR:", int((y > q3 + 1.5*(q3-q1)).sum()))
print("Skew price:", round(y.skew(), 3), "| skew log(price):", round(np.log(y).skew(), 3))

,corr_with_price
sqft_living,0.702
grade,0.667
sqft_above,0.606
sqft_living15,0.585
bathrooms,0.525
view,0.397
sqft_basement,0.324
bedrooms,0.315
lat,0.307
waterfront,0.266


Số giá nhà outlier theo IQR: 1146
Skew price: 4.024 | skew log(price): 0.428


## 3. Feature Engineering

Tạo 17 feature mới (không dùng target nên không rò rỉ): thời gian (`house_age`, `years_since_update`, `sale_year`...), tỉ lệ diện tích, phi tuyến/tương tác (`log_sqft_living`, `grade_sq`, `living_x_grade`), vị trí (`dist_center_km`) và mã hóa `zipcode` bằng Target Encoding (học trong pipeline, có cross-fitting).

In [5]:
# ---------- 2. Feature Engineering (khong dung target => khong ro ri) ----------
def engineer(d):
    f = d[BASE].copy()
    sale_year = d["date"].dt.year
    f["sale_year"] = sale_year; f["sale_month"] = d["date"].dt.month
    f["house_age"] = sale_year - d["yr_built"]
    f["is_renovated"] = (d["yr_renovated"] > 0).astype(int)
    f["years_since_update"] = sale_year - np.maximum(d["yr_built"], d["yr_renovated"])
    f["has_basement"] = (d["sqft_basement"] > 0).astype(int)
    f["total_rooms"] = d["bedrooms"] + d["bathrooms"]
    f["bath_per_bed"] = d["bathrooms"] / d["bedrooms"].clip(lower=1)
    f["living_lot_ratio"] = d["sqft_living"] / d["sqft_lot"]
    f["living_vs_neighbors"] = d["sqft_living"] / d["sqft_living15"]
    f["lot_vs_neighbors"] = d["sqft_lot"] / d["sqft_lot15"]
    f["log_sqft_living"] = np.log(d["sqft_living"]); f["log_sqft_lot"] = np.log(d["sqft_lot"])
    f["grade_sq"] = d["grade"] ** 2
    f["living_x_grade"] = d["sqft_living"] * d["grade"]
    f["view_waterfront"] = d["view"] + 2 * d["waterfront"]
    # khoang cach den trung tam Seattle (xap xi haversine, km)
    lat0, lon0 = 47.6062, -122.3321
    dlat = np.radians(d["lat"] - lat0); dlon = np.radians(d["long"] - lon0)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat0))*np.cos(np.radians(d["lat"]))*np.sin(dlon/2)**2
    f["dist_center_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))
    return f
X_base = df[BASE].copy()
X_fe = engineer(df)
NEW = [c for c in X_fe.columns if c not in BASE]
log("FE features:", X_base.shape[1], "->", X_fe.shape[1])
display(X_fe[NEW].head())

[   10s] FE features: 18 -> 35


,sale_year,sale_month,house_age,is_renovated,years_since_update,has_basement,total_rooms,bath_per_bed,living_lot_ratio,living_vs_neighbors,lot_vs_neighbors,log_sqft_living,log_sqft_lot,grade_sq,living_x_grade,view_waterfront,dist_center_km
0,2014,10,59,0,59,0,4.00,0.333333,0.208850,0.880597,1.000000,7.073270,8.639411,49,8260,0,11.972687
1,2014,12,63,1,23,1,5.25,0.750000,0.354874,1.520710,0.948030,7.851661,8.887653,49,17990,0,12.802819
2,2015,2,82,0,82,0,3.00,0.500000,0.077000,0.283088,1.240387,6.646391,9.210340,36,4620,0,16.416960
3,2014,12,49,0,49,1,7.00,0.750000,0.392000,1.441176,1.000000,7.580700,8.517193,49,13720,0,10.538233
4,2015,2,28,0,28,0,5.00,0.666667,0.207921,0.933333,1.076903,7.426549,8.997147,64,13440,0,21.553979


## 4. Chia train/test và định nghĩa pipeline

In [ ]:
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
ytr, yte = y.iloc[idx_tr], y.iloc[idx_te]
kf = KFold(5, shuffle=True, random_state=SEED)

# ---------- 3. Pipeline ----------
def prep(cols, te=True):
    """te=True: zipcode -> target encoding (hoc trong train, co cross-fitting); te=False: zipcode giu nhu so nguyen."""
    num = [c for c in cols if c != "zipcode" or not te]
    steps = [("num", StandardScaler(), num)]
    if "zipcode" in cols and te: steps.append(("zip", Pipeline([("te", TargetEncoder(target_type="continuous", random_state=SEED)), ("sc", StandardScaler())]), ["zipcode"]))
    return ColumnTransformer(steps)

def make(model, cols, log_target=True, selector=None, te=True):
    st = [("prep", prep(cols, te))]
    if selector is not None: st.append(("sel", selector))
    st.append(("model", model))
    p = Pipeline(st)
    return TransformedTargetRegressor(p, func=np.log, inverse_func=np.exp) if log_target else p

def metrics(yt, yp):
    return dict(MAE=mean_absolute_error(yt, yp), RMSE=np.sqrt(mean_squared_error(yt, yp)), R2=r2_score(yt, yp), MAPE=100*mean_absolute_percentage_error(yt, yp))

def run(name, model, X, cols=None, log_target=True, selector=None, cv=True, extra=None, te=True):
    cols = cols or list(X.columns)
    m = make(model, cols, log_target, selector, te)
    Xtr, Xte = X.iloc[idx_tr][cols], X.iloc[idx_te][cols]
    m.fit(Xtr, ytr); ptr, pte = m.predict(Xtr), m.predict(Xte)
    r = dict(Name=name, n_features=len(cols), **metrics(yte, pte), R2_train=r2_score(ytr, ptr))
    if cv:
        s = cross_validate(make(model, cols, log_target, selector, te), Xtr, ytr, cv=kf, scoring="r2", n_jobs=1)
        r["CV_R2"] = s["test_score"].mean(); r["CV_std"] = s["test_score"].std()
    if extra: r.update(extra)
    return r, pte, m

def models(fast=False):
    return {"Linear Regression": LinearRegression(), "Ridge": Ridge(alpha=1.0), "Lasso": Lasso(alpha=0.001, max_iter=5000),
            "KNN": KNeighborsRegressor(n_neighbors=10, weights="distance"),
            "Decision Tree": DecisionTreeRegressor(max_depth=10, min_samples_leaf=10, random_state=SEED),
            "Random Forest": RandomForestRegressor(n_estimators=100 if fast else 150, min_samples_leaf=2, max_features=0.5, n_jobs=-1, random_state=SEED),
            "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=SEED),
            "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=SEED)}
print("Train:", len(idx_tr), "| Test:", len(idx_te))

## 5. Giai đoạn A, B, C: Baseline → log(price) → Feature Engineering

- **A:** 18 feature gốc, dự đoán `price` trực tiếp (zipcode là số nguyên).
- **B:** như A nhưng huấn luyện trên `log(price)`.
- **C:** thêm toàn bộ 17 feature mới + target encoding zipcode.

In [ ]:
R = {}   # lưu kết quả từng giai đoạn
# ---- Giai doan A: Baseline (feature goc, y = price, khong FE) ----
rowsA = []
for n, m in models().items():
    r, _, _ = run(n, m, X_base, log_target=False, te=False); r["Stage"] = "A: Baseline (feature gốc, y = price)"; rowsA.append(r); log("A", n, round(r["R2"], 4))
R["A"] = rowsA

# ---- Giai doan B: log-target (van feature goc) ----
rowsB = []
for n, m in models().items():
    r, _, _ = run(n, m, X_base, log_target=True, te=False); r["Stage"] = "B: Feature gốc + log(price)"; rowsB.append(r); log("B", n, round(r["R2"], 4))
R["B"] = rowsB

# ---- Giai doan C: Feature Engineering (+ log target) ----
rowsC = []
for n, m in models().items():
    r, _, _ = run(n, m, X_fe, log_target=True); r["Stage"] = "C: Feature Engineering + log(price)"; rowsC.append(r); log("C", n, round(r["R2"], 4))
R["C"] = rowsC

## 6. Đóng góp của từng nhóm Feature Engineering

In [ ]:
# ---- Giai doan C2: dong gop tung nhom FE (Linear va HistGB) ----
groups = {"Gốc (18 feature)": [],
  "+ thời gian (sale_year, sale_month, house_age, is_renovated, years_since_update)": ["sale_year","sale_month","house_age","is_renovated","years_since_update"],
  "+ diện tích/tỉ lệ (total_rooms, bath_per_bed, living_lot_ratio, living_vs_neighbors, lot_vs_neighbors, has_basement)": ["total_rooms","bath_per_bed","living_lot_ratio","living_vs_neighbors","lot_vs_neighbors","has_basement"],
  "+ phi tuyến/tương tác (log_sqft_living, log_sqft_lot, grade_sq, living_x_grade, view_waterfront)": ["log_sqft_living","log_sqft_lot","grade_sq","living_x_grade","view_waterfront"],
  "+ vị trí (dist_center_km và mã hóa zipcode bằng target encoding)": ["dist_center_km"]}
cols = list(BASE); rowsG = []
for g, add in groups.items():
    cols = cols + add
    for mn in ["Linear Regression", "HistGradientBoosting"]:
        r, _, _ = run(mn, models()[mn], X_fe, cols=list(cols), cv=False, te=("dist_center_km" in cols)); r["Group"] = g; rowsG.append(r)
    log("C2", g[:30])
R["G"] = rowsG

## 7. Feature Selection (đặt trong pipeline nên cross-validation không bị rò rỉ)

So sánh 5 phương pháp: F-test/tương quan Top-15, Mutual Information Top-15, RFE, Lasso L1 và độ quan trọng của Random Forest.

In [ ]:
# ---- Giai doan D: Feature Selection (tren tap FE, trong pipeline => khong ro ri) ----
K = 15
sel = {"Không chọn (toàn bộ FE)": None,
       "Tương quan/F-test Top-15 (SelectKBest f_regression)": SelectKBest(f_regression, k=K),
       "Mutual Information Top-15": SelectKBest(lambda X_, y_: mutual_info_regression(X_, y_, random_state=SEED, n_neighbors=5), k=K),
       "RFE (Linear Regression) 15 feature": RFE(LinearRegression(), n_features_to_select=K),
       "Lasso L1 (SelectFromModel)": SelectFromModel(Lasso(alpha=0.003, max_iter=5000)),
       "Random Forest importance Top-15": SelectFromModel(RandomForestRegressor(n_estimators=60, max_depth=12, n_jobs=-1, random_state=SEED), max_features=K, threshold=-np.inf)}
rowsD = []; selected = {}
for sn, s in sel.items():
    for mn in ["Linear Regression", "HistGradientBoosting"]:
        r, _, m = run(mn, models()[mn], X_fe, selector=s, extra=dict(Selector=sn)); rowsD.append(r)
    if s is not None:
        fitted = make(models()["Linear Regression"], list(X_fe.columns), True, s).fit(X_fe.iloc[idx_tr], ytr)
        pipe = fitted.regressor_; names = [c for c in pipe.named_steps["prep"].get_feature_names_out()]
        mask = pipe.named_steps["sel"].get_support(); selected[sn] = [n.split("__")[1] for n, k in zip(names, mask) if k]
    log("D", sn)
R["D"] = rowsD; R["selected"] = selected

## 8. Mô hình cuối: Feature Engineering + Feature Selection + tinh chỉnh siêu tham số

In [ ]:
# ---- Giai doan E: mo hinh cuoi cung (FE + selection tot nhat) + tinh chinh ----
best_sel_name = max(sel.keys(), key=lambda k: np.mean([r["CV_R2"] for r in rowsD if r["Selector"] == k]))
log("Best selector by mean CV R2:", best_sel_name)
R["best_selector"] = best_sel_name
feat_final = selected.get(best_sel_name, list(X_fe.columns))
rowsE = []; preds = {}
for n, m in models().items():
    r, p, _ = run(n, m, X_fe, cols=feat_final); r["Stage"] = "E: FE + Selection (mặc định)"; rowsE.append(r); preds[n + "|default"] = p; log("E", n, round(r["R2"], 4))
R["E"] = rowsE

grids = {"Ridge": (Ridge(), {"model__alpha": [0.01, 0.1, 1, 10, 100]}, "grid"),
         "Lasso": (Lasso(max_iter=5000), {"model__alpha": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]}, "grid"),
         "Random Forest": (RandomForestRegressor(n_jobs=-1, random_state=SEED), {"model__n_estimators": [150, 250], "model__max_depth": [None, 20], "model__min_samples_leaf": [1, 2, 4], "model__max_features": [0.33, 0.5, 0.8]}, "rand"),
         "HistGradientBoosting": (HistGradientBoostingRegressor(random_state=SEED), {"model__max_iter": [300, 500], "model__learning_rate": [0.03, 0.05, 0.08], "model__max_depth": [None, 6, 10], "model__min_samples_leaf": [10, 20, 40], "model__l2_regularization": [0, 1]}, "rand")}
rowsT = []; bestp = {}
for n, (m, g, kind) in grids.items():
    base = Pipeline([("prep", prep(feat_final)), ("model", m)])
    ttr = TransformedTargetRegressor(base, func=np.log, inverse_func=np.exp)
    g2 = {"regressor__" + k: v for k, v in g.items()}
    Xtr = X_fe.iloc[idx_tr][feat_final]
    s = (GridSearchCV(ttr, g2, cv=3, scoring="neg_mean_squared_error", n_jobs=1) if kind == "grid"
         else RandomizedSearchCV(ttr, g2, n_iter=8, cv=3, scoring="neg_mean_squared_error", n_jobs=1, random_state=SEED)).fit(Xtr, ytr)
    bestp[n] = {k.replace("regressor__model__", ""): v for k, v in s.best_params_.items()}
    pte = s.predict(X_fe.iloc[idx_te][feat_final]); ptr = s.predict(Xtr)
    cvs = cross_validate(s.best_estimator_, Xtr, ytr, cv=kf, scoring="r2")
    r = dict(Name=n, n_features=len(feat_final), **metrics(yte, pte), R2_train=r2_score(ytr, ptr), CV_R2=cvs["test_score"].mean(), CV_std=cvs["test_score"].std(), Stage="F: FE + Selection + tinh chỉnh")
    rowsT.append(r); preds[n + "|tuned"] = pte; log("F", n, bestp[n], round(r["R2"], 4))
R["F"] = rowsT; R["best_params"] = bestp

# ket qua tong hop: tuned neu co, con lai default
final = {r["Name"]: r for r in rowsE}
for r in rowsT: final[r["Name"]] = r
R["final"] = list(final.values())
for n in final: preds[n + "|final"] = preds[n + ("|tuned" if n in bestp else "|default")]

In [ ]:
preds_final = {k: v for k, v in preds.items() if k.endswith("|final")}
yte_arr = yte.values

In [ ]:
# feature importance (HGB tuned bang permutation tren test) 
from sklearn.inspection import permutation_importance
hp = bestp["HistGradientBoosting"]
mh = make(HistGradientBoostingRegressor(random_state=SEED, **hp), feat_final, True).fit(X_fe.iloc[idx_tr][feat_final], ytr)
pi = permutation_importance(mh, X_fe.iloc[idx_te][feat_final], yte, n_repeats=5, random_state=SEED, scoring="r2")
imp = pd.Series(pi.importances_mean, index=feat_final).sort_values(ascending=False); R["perm_importance"] = imp.round(4).to_dict()

In [ ]:
# EDA so lieu cho bao cao
eda = {"n_raw": n0, "n_clean": len(df), "price": y.describe().to_dict(), "skew_price": float(y.skew()), "skew_logprice": float(np.log(y).skew()),
       "corr_price": df[BASE].assign(price=y).corr()["price"].drop("price").sort_values(key=abs, ascending=False).round(3).to_dict(),
       "price_outliers_iqr": int((y > y.quantile(.75) + 1.5 * (y.quantile(.75) - y.quantile(.25))).sum()),
       "dup_id": int(df["id"].duplicated().sum()), "n_train": len(idx_tr), "n_test": len(idx_te), "new_features": NEW}
R["eda"] = eda
fe_corr = X_fe.assign(lp=np.log(y)).corr()["lp"].drop("lp"); R["fe_corr_logprice"] = fe_corr.sort_values(key=abs, ascending=False).round(3).to_dict()

## 9. So sánh kết quả

In [ ]:
cols = ["n_features", "MAE", "RMSE", "R2", "MAPE", "R2_train", "CV_R2", "CV_std"]
for key, title in [("A", "A: Baseline (feature gốc, y = price)"), ("B", "B: feature gốc + log(price)"), ("C", "C: Feature Engineering + log(price)"), ("E", "E: FE + Feature Selection (tham số mặc định)")]:
    print(title); display(pd.DataFrame(R[key]).set_index("Name")[cols].round(4))
print("Tinh chỉnh siêu tham số:", R["best_params"])
print("Bảng cuối (tinh chỉnh nếu có, còn lại mặc định):")
display(pd.DataFrame(R["final"]).set_index("Name")[cols].round(4))

### Biểu đồ

In [ ]:
# Biểu đồ tổng hợp (cần: R, preds_final, yte_arr, df, y)
order = ["Linear Regression", "Ridge", "Lasso", "KNN", "Decision Tree", "Random Forest", "Gradient Boosting", "HistGradientBoosting"]
short = ["Linear", "Ridge", "Lasso", "KNN", "D.Tree", "RF", "GBR", "HGB"]
def tbl(key):
    return pd.DataFrame(R[key]).set_index("Name").reindex(order)

# 1) Phân phối giá nhà: gốc và log
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(y / 1e6, bins=60, color="#6f9bc8", edgecolor="white"); ax[0].set_title(f"Phân phối price (skew = {R['eda']['skew_price']:.2f})"); ax[0].set_xlabel("price (triệu USD)"); ax[0].set_ylabel("Tần suất")
ax[1].hist(np.log(y), bins=60, color="#1f4e79", edgecolor="white"); ax[1].set_title(f"Phân phối log(price) (skew = {R['eda']['skew_logprice']:.2f})"); ax[1].set_xlabel("log(price)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("h1_price_dist.png", dpi=150) if SAVE else None; plt.show()

# 2) Tương quan với price
c = pd.Series(R["eda"]["corr_price"]).sort_values()
fig, a = plt.subplots(figsize=(7, 5.2)); a.barh(c.index, c.values, color=["#c0392b" if v < 0 else "#2e6da4" for v in c.values])
a.set_xlabel("Hệ số tương quan Pearson với price"); a.set_title("Tương quan của các feature gốc với price"); a.grid(alpha=.3, axis="x")
plt.tight_layout(); plt.savefig("h2_corr.png", dpi=150) if SAVE else None; plt.show()

# 3) So sánh các giai đoạn theo R2 test
stages = {"A: gốc, y=price": tbl("A"), "B: gốc + log(price)": tbl("B"), "C: + Feature Engineering": tbl("C"), "E: + Feature Selection": tbl("E")}
fig, a = plt.subplots(figsize=(13, 4.6)); x = np.arange(len(order)); w = .2; cols = ["#b0c4de", "#7fa6cf", "#3f78b0", "#1f4e79"]
for i, (k, t) in enumerate(stages.items()):
    a.bar(x + (i - 1.5) * w, t["R2"], w, label=k, color=cols[i])
a.set_xticks(x); a.set_xticklabels(short); a.set_ylabel("R² (tập test)"); a.set_title("R² test của 8 mô hình qua các giai đoạn pipeline"); a.set_ylim(0, 1.1); a.legend(fontsize=8, ncol=4, loc="upper center"); a.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.savefig("h3_stages.png", dpi=150) if SAVE else None; plt.show()

# 4) Đóng góp từng nhóm Feature Engineering
G = pd.DataFrame(R["G"]); labs = ["Gốc", "+ thời gian", "+ diện tích/tỉ lệ", "+ phi tuyến", "+ vị trí (dist, zipcode TE)"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, mn in zip(ax, ["Linear Regression", "HistGradientBoosting"]):
    v = G[G.Name == mn]["R2"].values; a.plot(labs, v, "o-", color="#1f4e79"); a.set_title(f"R² test theo nhóm feature: {mn}"); a.grid(alpha=.3); a.tick_params(axis="x", rotation=25)
    for i, t in enumerate(v): a.text(i, t, f"{t:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig("h4_fe_groups.png", dpi=150) if SAVE else None; plt.show()

# 5) So sánh các phương pháp Feature Selection
D = pd.DataFrame(R["D"]); sels = list(dict.fromkeys(D.Selector)); sl = ["Không chọn\n(35)", "F-test\nTop-15", "Mutual Info\nTop-15", "RFE\n15", "Lasso L1\n(21)", "RF imp.\nTop-15"]
fig, a = plt.subplots(figsize=(11, 4.4)); x = np.arange(len(sels)); w = .38
for i, mn in enumerate(["Linear Regression", "HistGradientBoosting"]):
    a.bar(x + (i - .5) * w, [D[(D.Selector == s) & (D.Name == mn)].R2.values[0] for s in sels], w, label=mn, color=["#7fa6cf", "#1f4e79"][i])
a.set_xticks(x); a.set_xticklabels(sl, fontsize=8); a.set_ylabel("R² (tập test)"); a.set_ylim(0.5, 1); a.set_title("Hiệu năng theo phương pháp chọn feature"); a.legend(fontsize=8); a.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.savefig("h5_selection.png", dpi=150) if SAVE else None; plt.show()

# 6) So sánh cuối cùng
F = pd.DataFrame(R["final"]).set_index("Name").reindex(order)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
for a, (k, t, f) in zip(ax, [("MAE", "MAE (USD, thấp hơn tốt hơn)", "{:.0f}"), ("RMSE", "RMSE (USD, thấp hơn tốt hơn)", "{:.0f}"), ("R2", "R² (cao hơn tốt hơn)", "{:.3f}")]):
    b = a.bar(short, F[k], color="#1f4e79"); a.set_title(t); a.set_ylim(0.75, 0.95) if k == "R2" else None; a.grid(alpha=.3, axis="y"); a.tick_params(axis="x", rotation=30)
    for r_, v in zip(b, F[k]): a.text(r_.get_x() + r_.get_width() / 2, v, f.format(v), ha="center", va="bottom", fontsize=7)
plt.tight_layout(); plt.savefig("h6_final.png", dpi=150) if SAVE else None; plt.show()

# 7) Actual vs Predicted
sel_models = ["Linear Regression", "Random Forest", "Gradient Boosting", "HistGradientBoosting"]
fig, ax = plt.subplots(2, 2, figsize=(10, 9))
for a, n in zip(ax.ravel(), sel_models):
    p = preds_final[n + "|final"]; a.scatter(yte_arr / 1e6, p / 1e6, alpha=.35, s=8); lim = [0, 4]; a.plot(lim, lim, "--", c="r"); a.set_xlim(lim); a.set_ylim(lim)
    a.set_title(f"{n}\nR² = {r2_score(yte_arr, p):.3f}", fontsize=10); a.set_xlabel("Giá thực tế (triệu USD)"); a.set_ylabel("Giá dự đoán (triệu USD)"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("h7_actual_pred.png", dpi=150) if SAVE else None; plt.show()

# 8) Residual của mô hình tốt nhất
p = preds_final["Gradient Boosting|final"]; res_ = yte_arr - p
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(p / 1e6, res_ / 1e3, alpha=.3, s=8); ax[0].axhline(0, ls="--", c="r"); ax[0].set_xlabel("Giá dự đoán (triệu USD)"); ax[0].set_ylabel("Residual (nghìn USD)"); ax[0].set_title("Residual plot: Gradient Boosting"); ax[0].grid(alpha=.3)
ax[1].hist(res_ / 1e3, bins=80, color="#6f9bc8", edgecolor="white"); ax[1].set_xlim(-600, 600); ax[1].set_xlabel("Residual (nghìn USD)"); ax[1].set_title(f"Phân phối residual (mean = {res_.mean()/1e3:.1f}k)"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("h8_residual.png", dpi=150) if SAVE else None; plt.show()

# 9) Độ quan trọng feature (permutation importance, HGB đã tinh chỉnh)
imp = pd.Series(R["perm_importance"]).sort_values().tail(12)
fig, a = plt.subplots(figsize=(7, 4.6)); a.barh(imp.index, imp.values, color="#1f4e79"); a.set_title("Permutation importance (giảm R² khi xáo trộn feature)\nHistGradientBoosting trên tập test"); a.grid(alpha=.3, axis="x")
plt.tight_layout(); plt.savefig("h9_importance.png", dpi=150) if SAVE else None; plt.show()

## 10. Nhận xét

- **Baseline (A):** các mô hình tuyến tính chỉ đạt R² ≈ 0,70; các mô hình dựa trên cây mạnh hơn rõ (Random Forest 0,86; Gradient Boosting 0,88), vì giá nhà phụ thuộc phi tuyến vào vị trí và chất lượng.
- **log(price) (B):** giúp Gradient Boosting và HistGradientBoosting (R² tăng lên 0,89 đến 0,90), nhưng làm R² trên thang giá gốc của Linear Regression giảm (0,50) vì hàm `exp` khuếch đại sai số ở các căn nhà đắt (có nhà bị dự đoán vượt 10 triệu USD).
- **Feature Engineering (C):** cải thiện lớn nhất cho mô hình tuyến tính (R² 0,50 → 0,84), chủ yếu nhờ target encoding `zipcode` và `dist_center_km`. Gradient Boosting đạt tốt nhất: R² ≈ 0,916, RMSE ≈ 112,5k USD, MAE ≈ 63,9k USD.
- **Feature Selection (E):** giảm 35 → 21 feature (Lasso L1) mà hiệu năng gần như không đổi (Linear 0,842; Gradient Boosting 0,910 so với 0,916). Mutual Information và RF importance chọn kém hơn ở mô hình tuyến tính.
- **Tinh chỉnh:** cải thiện nhỏ cho Random Forest (0,886 → 0,887) và HistGradientBoosting (0,904 → 0,908).
- **Kết luận:** nhóm Gradient Boosting (GBR/HGB) tốt nhất, sau đó là Random Forest; Linear/Ridge/Lasso ≈ 0,84; KNN và Decision Tree thấp nhất. Feature quan trọng nhất: `living_x_grade`, `zipcode`, `dist_center_km`, `grade`, `lat`.